# Flipkart Traffic Demand — Per-RoadType LightGBM (The 92+ Architecture)
**Model:** Separate LightGBM trained on 100% of data (Day 48 + Day 49), one per road type.
**Upgrades:** Geohash Lat/Lon Decoding, T-24 Hourly Lookups, and Leak-Proof 5-Fold OOF Encoding.

## 1. Imports & Config

In [14]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import lightgbm as lgb
import pygeohash as pgh  # pip install pygeohash
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

SEED = 42
np.random.seed(SEED)

def competition_score(actual, predicted):
    return max(0, 100 * r2_score(actual, predicted))

## 2. Load Data

In [2]:
df_raw = pd.read_csv("raw.csv")
df = df_raw.copy()
print(f"Shape: {df.shape}")
print(df.head())

Shape: (77299, 11)
   Index geohash  day timestamp  demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.0488          NaN              1   
1      1  qp02zt   48       0:0  0.1185  Residential              3   
2      2  qp08bj   48       0:0  0.0271  Residential              1   
3      3  qp08gt   48       0:0  0.0033  Residential              1   
4      4  qp02zq   48       0:0  0.0108  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes      31.1046   Sunny  
2   Not Allowed        No      25.9193   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No      10.8037   Rainy  


## 3. Data Cleaning

In [3]:
# ── RoadType: impute from most common value per geohash ──────────────────────
geo_roadtype_mode = (
    df.groupby("geohash")["RoadType"]
    .agg(lambda x: x.mode()[0] if x.notna().any() else np.nan)
)
df["RoadType"] = df["RoadType"].fillna(df["geohash"].map(geo_roadtype_mode))
df["RoadType"] = df["RoadType"].fillna(df["RoadType"].mode()[0])

# ── Weather: small missingness — treat as its own category ───────────────────
df["Weather"] = df["Weather"].fillna("Unknown")

# ── Temperature: median per (geohash, hour), fallback to global median ────────
df["_hour_tmp"] = df["timestamp"].str.split(":").str[0].astype(int)
geo_hour_temp_median = df.groupby(["geohash", "_hour_tmp"])["Temperature"].median()

def fill_temperature(row):
    if pd.isnull(row["Temperature"]):
        return geo_hour_temp_median.get((row["geohash"], row["_hour_tmp"]), np.nan)
    return row["Temperature"]

df["Temperature"] = df.apply(fill_temperature, axis=1)
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
df.drop(columns=["_hour_tmp"], inplace=True)

assert df.isnull().sum().sum() == 0, "Nulls remain after cleaning!"
print("✓ No nulls remaining.")

✓ No nulls remaining.


## 4. Feature Engineering

In [4]:
# %% [markdown]
# ## 4. Feature Engineering & Spatio-Temporal Clustering

# %%
from sklearn.cluster import KMeans

# ── Parse timestamps ─────────────────────────────────────────────────────────
df["hour"]         = df["timestamp"].str.split(":").str[0].astype(int)
df["minute"]       = df["timestamp"].str.split(":").str[1].astype(int)
df["time_minutes"] = df["hour"] * 60 + df["minute"]

# ── Cyclical time encoding (so 23:45 is close to 00:00) ──────────────────────
PERIOD = 1440
df["time_sin"] = np.sin(2 * np.pi * df["time_minutes"] / PERIOD)
df["time_cos"] = np.cos(2 * np.pi * df["time_minutes"] / PERIOD)

# ── Sort chronologically before computing anything else ──────────────────────
df = df.sort_values(["geohash", "day", "time_minutes"]).reset_index(drop=True)

# ── Continuous Spatial Decoding & Macro Grids ────────────────────────────────
df["lat"] = df["geohash"].apply(lambda x: pgh.decode(x)[0])
df["lon"] = df["geohash"].apply(lambda x: pgh.decode(x)[1])
df["geo5"] = df["geohash"].str[:5]
df["geo4"] = df["geohash"].str[:4]  # <-- Macro-Resolution District

# ── K-Means Traffic Archetypes ───────────────────────────────────────────────
print("Clustering Spatio-Temporal Archetypes...")
# 1. Pivot the data to get a 24-hour traffic curve for each geohash
pivot_df = df[df["day"] == 48].pivot_table(
    index="geohash", 
    columns="hour", 
    values="demand", 
    aggfunc="mean"
).fillna(0)

# 2. Cluster the geohashes into 5 distinct "Traffic Archetypes"
kmeans = KMeans(n_clusters=5, random_state=SEED, n_init=10)
pivot_df["traffic_archetype"] = kmeans.fit_predict(pivot_df)

# 3. Map the archetype back to the main dataframe
archetype_map = pivot_df["traffic_archetype"].to_dict()
df["traffic_archetype"] = df["geohash"].map(archetype_map).fillna(-1).astype(int)

Clustering Spatio-Temporal Archetypes...


## 5. Full-Dataset Prep & 5-Fold OOF Target Encoding
We encode specific interaction features safely to prevent leakage.
For the test set, these features will act as our exact T-24 momentum lookups.

In [5]:
# %% [markdown]
# ## 5. Full-Dataset Prep & 5-Fold OOF Target Encoding

# %%
df_full = df.copy()

def create_oof_encodings(df, cols_to_encode, target_col="demand", n_splits=5, smoothing=15):
    df_out = df.copy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    global_mean = df[target_col].mean()
    mappings = {}

    for col in cols_to_encode:
        name = "_".join(col) + "_encoded" if isinstance(col, tuple) else col + "_encoded"
        df_out[name] = np.nan
        group_cols = list(col) if isinstance(col, tuple) else [col]

        # 1. OOF mapping for training data (prevents target leakage)
        for trn_idx, val_idx in kf.split(df):
            trn_data = df.iloc[trn_idx]
            val_data = df.iloc[val_idx].copy()

            stats = trn_data.groupby(group_cols)[target_col].agg(["mean", "count"])
            stats["smoothed"] = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)

            val_data = val_data.merge(stats[["smoothed"]], on=group_cols, how="left")
            df_out.loc[val_idx, name] = val_data["smoothed"].fillna(global_mean).values

        # 2. 100% mapping for the test data
        stats_full = df.groupby(group_cols)[target_col].agg(["mean", "count"])
        stats_full["smoothed"] = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
        mappings[name] = stats_full["smoothed"].to_dict()

    return df_out, mappings, global_mean

# Apply OOF Encodings (Added geo4)
interactions = ["geohash", ("geo5", "hour"), ("geo4", "hour"), ("geohash", "hour")]
df_full, target_mappings, global_demand_mean = create_oof_encodings(df_full, interactions)

import geohash2
print("Calculating Spatial Spillovers (The Geohash Ring)...")
global_geo_means = df_full.groupby('geohash')['demand'].mean().to_dict()

def get_neighbor_demand(ghash):
    try:
        neighbors = geohash2.neighbors(ghash)
        if isinstance(neighbors, dict):
            neighbors = list(neighbors.values())
        demands = [global_geo_means.get(n, global_demand_mean) for n in neighbors]
        return np.mean(demands)
    except:
        return global_demand_mean

df_full["surrounding_ring_demand"] = df_full["geohash"].apply(get_neighbor_demand)

# ── Binary categorical encodings ─────────────────────────────────────────────
df_full["LargeVehicles_enc"] = (df_full["LargeVehicles"] == "Allowed").astype(int)
df_full["Landmarks_enc"]     = (df_full["Landmarks"] == "Yes").astype(int)

# ── One-hot encode RoadType, Weather, and Archetypes ─────────────────────────
df_full = pd.get_dummies(df_full, columns=["RoadType"], drop_first=True)
df_full = pd.get_dummies(df_full, columns=["Weather"],  drop_first=True)
df_full = pd.get_dummies(df_full, columns=["traffic_archetype"], drop_first=False)

# ── Yeo-Johnson target transform ─────────────────────────────────────────────
pt_yeo = PowerTransformer(method="yeo-johnson", standardize=False)
df_full["yeo_demand"] = pt_yeo.fit_transform(df_full[["demand"]]).flatten()
yeo_min = df_full["yeo_demand"].min()
yeo_max = df_full["yeo_demand"].max()

df_full["geohash_cat"] = df_full["geohash"].astype("category")
known_cats = df_full["geohash_cat"].cat.categories

print(f"df_full shape: {df_full.shape}")

Calculating Spatial Spillovers (The Geohash Ring)...
df_full shape: (77299, 39)


## 6. Feature List

In [6]:
# %% [markdown]
# ## 6. Feature List

# %%
FEATURES = [
    "time_sin", "time_cos", "time_minutes", "hour",
    "lat", "lon",  
    "NumberofLanes", "LargeVehicles_enc", "Landmarks_enc",
    *[c for c in df_full.columns if c.startswith("RoadType_")],
    *[c for c in df_full.columns if c.startswith("Weather_")],
    *[c for c in df_full.columns if c.startswith("traffic_archetype_")], # <-- Archetypes
    "Temperature",
    "geohash_encoded", 
    "geohash_hour_encoded", 
    "geo5_hour_encoded",    
    "geo4_hour_encoded", # <-- Macro Resolution
    "surrounding_ring_demand"
]

FEATURES_LGB = FEATURES + ["geohash_cat"]

missing = [f for f in FEATURES_LGB if f not in df_full.columns]
assert len(missing) == 0, f"Missing features: {missing}"
print(f"Features ({len(FEATURES_LGB)}): {FEATURES_LGB}")

Features (28): ['time_sin', 'time_cos', 'time_minutes', 'hour', 'lat', 'lon', 'NumberofLanes', 'LargeVehicles_enc', 'Landmarks_enc', 'RoadType_Residential', 'RoadType_Street', 'Weather_Rainy', 'Weather_Snowy', 'Weather_Sunny', 'Weather_Unknown', 'traffic_archetype_-1', 'traffic_archetype_0', 'traffic_archetype_1', 'traffic_archetype_2', 'traffic_archetype_3', 'traffic_archetype_4', 'Temperature', 'geohash_encoded', 'geohash_hour_encoded', 'geo5_hour_encoded', 'geo4_hour_encoded', 'surrounding_ring_demand', 'geohash_cat']


## 7. LightGBM Hyperparameters

In [ ]:
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       127,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "verbose":          -1,
    "seed":             SEED,
}

## 8. Train Per-RoadType Models on Full Data

In [ ]:
road_types = ["Highway", "Residential", "Street"]
models_per_road = {}

for road_type in road_types:
    print(f"\nTraining: {road_type}")

    res_mask = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_mask = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)

    if road_type == "Residential":
        mask = res_mask
    elif road_type == "Street":
        mask = str_mask
    else:  # Highway is the dropped reference category
        mask = ~(res_mask | str_mask)

    sub = df_full[mask]
    print(f"  Rows: {len(sub):,}")

    lgb_data = lgb.Dataset(
        sub[FEATURES_LGB], label=sub["yeo_demand"],
        categorical_feature=["geohash_cat"], free_raw_data=False,
    )
    model = lgb.train(
        params          = lgb_params,
        train_set       = lgb_data,
        num_boost_round = 200,
    )
    models_per_road[road_type] = model
    print(f"  ✓ Done")

print("\nAll models trained.")


Training: Highway
  Rows: 3,593
  ✓ Done

Training: Residential
  Rows: 69,789
  ✓ Done

Training: Street
  Rows: 3,917
  ✓ Done

All models trained.


## 9. Sanity Check — Validation Score on Day 49

In [9]:
df_val_check = df_full[df_full["day"] == 49].copy()
y_val_raw = df_val_check["demand"].values

y_pred_check = np.zeros(len(df_val_check))

res_mask_v = df_val_check.get("RoadType_Residential", pd.Series(0, index=df_val_check.index)).astype(bool)
str_mask_v = df_val_check.get("RoadType_Street",      pd.Series(0, index=df_val_check.index)).astype(bool)
hwy_mask_v = ~(res_mask_v | str_mask_v)

for road_type, mask in [("Highway", hwy_mask_v), ("Street", str_mask_v), ("Residential", res_mask_v)]:
    if road_type not in models_per_road:
        continue
    idx = df_val_check.index[mask]
    if len(idx) == 0:
        continue
    yeo_preds = models_per_road[road_type].predict(df_val_check.loc[idx, FEATURES_LGB])
    yeo_preds = np.clip(yeo_preds, yeo_min, yeo_max)
    y_pred_check[mask.values] = np.clip(
        pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
    )

score = competition_score(y_val_raw, y_pred_check)
print(f"Day 49 indicative score: {score:.4f}")

Day 49 indicative score: 87.9474


## 10. Test Pipeline Helper Functions

In [10]:
def align_columns(df_test_encoded, train_columns):
    """Ensure test matrix has exactly the same columns as training, in order."""
    for col in train_columns:
        if col not in df_test_encoded.columns:
            print(f"  [WARNING] '{col}' missing from test — adding as zeros.")
            df_test_encoded[col] = 0
    extra = set(df_test_encoded.columns) - set(train_columns)
    if extra:
        df_test_encoded.drop(columns=list(extra), inplace=True)
    return df_test_encoded[train_columns]

## 11. Full Test Pipeline

In [11]:
# %% [markdown]
# ## 11. Full Test Pipeline

# %%
def run_test_pipeline(
    test_path, df_full, target_mappings, global_demand_mean,
    models_per_road, pt_yeo, FEATURES_LGB, known_cats,
    yeo_min, yeo_max
):
    print("Step 1: Loading test data...")
    df_test = pd.read_csv(test_path)
    print(f"  Shape: {df_test.shape}")

    print("\nStep 2: Cleaning...")
    # RoadType — impute from geohash mode seen in training
    res_mask_tr = df_full.get("RoadType_Residential", pd.Series(0, index=df_full.index)).astype(bool)
    str_mask_tr = df_full.get("RoadType_Street",      pd.Series(0, index=df_full.index)).astype(bool)
    temp_rt = pd.Series("Highway", index=df_full.index)
    temp_rt.loc[res_mask_tr] = "Residential"
    temp_rt.loc[str_mask_tr] = "Street"
    geo_rt_mode = temp_rt.groupby(df_full["geohash"]).agg(lambda x: x.mode()[0])
    df_test["RoadType"] = (
        df_test["RoadType"]
        .fillna(df_test["geohash"].map(geo_rt_mode))
        .fillna("Highway")
    )

    # Weather — unknown for any nulls
    df_test["Weather"] = df_test["Weather"].fillna("Unknown")

    # Temperature — (geohash, hour) median from training, global fallback
    df_test["_hour_tmp"] = df_test["timestamp"].str.split(":").str[0].astype(int)
    geo_hr_temp = df_full.groupby(["geohash", "hour"])["Temperature"].median()
    df_test["Temperature"] = df_test.apply(
        lambda r: geo_hr_temp.get((r["geohash"], r["_hour_tmp"]), np.nan)
        if pd.isnull(r.get("Temperature", np.nan)) else r.get("Temperature", np.nan),
        axis=1
    ).fillna(df_full["Temperature"].median())
    df_test.drop(columns=["_hour_tmp"], inplace=True)

    print("\nStep 3: Feature engineering...")
    df_test["hour"]         = df_test["timestamp"].str.split(":").str[0].astype(int)
    df_test["minute"]       = df_test["timestamp"].str.split(":").str[1].astype(int)
    df_test["time_minutes"] = df_test["hour"] * 60 + df_test["minute"]
    PERIOD = 1440
    df_test["time_sin"] = np.sin(2 * np.pi * df_test["time_minutes"] / PERIOD)
    df_test["time_cos"] = np.cos(2 * np.pi * df_test["time_minutes"] / PERIOD)

    # Apply Lat/Lon/Geo Decoding
    df_test["lat"] = df_test["geohash"].apply(lambda x: pgh.decode(x)[0])
    df_test["lon"] = df_test["geohash"].apply(lambda x: pgh.decode(x)[1])
    df_test["geo5"] = df_test["geohash"].str[:5]
    df_test["geo4"] = df_test["geohash"].str[:4]

    # Map Archetype
    df_test["traffic_archetype"] = df_test["geohash"].map(archetype_map).fillna(-1).astype(int)

    print("Step 2.5: Calculating Spatial Spillovers for Test Set...")
    df_test["surrounding_ring_demand"] = df_test["geohash"].apply(get_neighbor_demand)

    # Apply Mappings
    df_test["geohash_encoded"] = df_test["geohash"].map(target_mappings["geohash_encoded"]).fillna(global_demand_mean)
    df_test["geo5_hour_encoded"] = df_test.set_index(["geo5", "hour"]).index.map(target_mappings["geo5_hour_encoded"]).fillna(global_demand_mean)
    df_test["geo4_hour_encoded"] = df_test.set_index(["geo4", "hour"]).index.map(target_mappings["geo4_hour_encoded"]).fillna(global_demand_mean)
    df_test["geohash_hour_encoded"] = df_test.set_index(["geohash", "hour"]).index.map(target_mappings["geohash_hour_encoded"]).fillna(global_demand_mean)

    df_test["LargeVehicles_enc"] = (df_test["LargeVehicles"] == "Allowed").astype(int)
    df_test["Landmarks_enc"]     = (df_test["Landmarks"] == "Yes").astype(int)
    
    # Categoricals (Including Archetype Dummies)
    df_test = pd.get_dummies(df_test, columns=["RoadType"], drop_first=True)
    df_test = pd.get_dummies(df_test, columns=["Weather"],  drop_first=True)
    df_test = pd.get_dummies(df_test, columns=["traffic_archetype"], drop_first=False)

    # Align columns to training schema
    feat_no_cat = [f for f in FEATURES_LGB if f != "geohash_cat"]
    X_test = align_columns(df_test.copy(), feat_no_cat)
    X_test["geohash_cat"] = pd.Categorical(df_test["geohash"], categories=known_cats)

    print("\nStep 4: Predicting per road type...")
    y_pred = np.zeros(len(df_test))

    res_mask = df_test.get("RoadType_Residential", pd.Series(0, index=df_test.index)).astype(bool)
    str_mask = df_test.get("RoadType_Street",      pd.Series(0, index=df_test.index)).astype(bool)
    hwy_mask = ~(res_mask | str_mask)

    for road_type, mask in [("Highway", hwy_mask), ("Street", str_mask), ("Residential", res_mask)]:
        if road_type not in models_per_road:
            continue
        idx = df_test.index[mask]
        if len(idx) == 0:
            continue
        yeo_preds = models_per_road[road_type].predict(X_test.loc[idx])
        yeo_preds = np.clip(yeo_preds, yeo_min, yeo_max)
        y_pred[mask.values] = np.clip(
            pt_yeo.inverse_transform(yeo_preds.reshape(-1, 1)).flatten(), 0, 1
        )
        print(f"  {road_type}: {mask.sum()} rows predicted")

    print("\nStep 5: Building submission...")
    submission = pd.DataFrame({"Index": df_test["Index"], "demand": y_pred})

    assert submission["demand"].isna().sum() == 0, "NaN predictions!"
    assert (submission["demand"] >= 0).all(),      "Negative predictions!"
    assert (submission["demand"] <= 1).all(),      "Predictions above 1!"
    assert len(submission) == len(df_test),        "Row count mismatch!"

    submission.to_csv("submission.csv", index=False)
    print(f"  ✓ submission.csv saved  ({len(submission):,} rows)")
    print(f"\n  demand stats:")
    print(submission["demand"].describe().round(4))
    return submission

## 12. Generate Submission

In [12]:
submission = run_test_pipeline(
    test_path          = "test.csv",
    df_full            = df_full,
    target_mappings    = target_mappings,
    global_demand_mean = global_demand_mean,
    models_per_road    = models_per_road,
    pt_yeo             = pt_yeo,
    FEATURES_LGB       = FEATURES_LGB,
    known_cats         = known_cats,
    yeo_min            = yeo_min,
    yeo_max            = yeo_max,
)

Step 1: Loading test data...
  Shape: (41778, 10)

Step 2: Cleaning...

Step 3: Feature engineering...
Step 2.5: Calculating Spatial Spillovers for Test Set...

Step 4: Predicting per road type...
  Highway: 4285 rows predicted
  Street: 3411 rows predicted
  Residential: 34082 rows predicted

Step 5: Building submission...
  ✓ submission.csv saved  (41,778 rows)

  demand stats:
count   41778.0000
mean        0.1192
std         0.1678
min         0.0028
25%         0.0236
50%         0.0498
75%         0.1328
max         1.0000
Name: demand, dtype: float64


## 13. Submission Audit

In [13]:
sub = pd.read_csv("submission.csv")
print(f"  Rows:        {len(sub):,}")
print(f"  Columns:     {sub.columns.tolist()}")
print(f"  NaNs:        {sub.isnull().sum().values}")
print(f"  Min demand:  {sub['demand'].min():.6f}  (must be >= 0)")
print(f"  Max demand:  {sub['demand'].max():.6f}  (must be <= 1)")
print()
print(sub.head(10))

  Rows:        41,778
  Columns:     ['Index', 'demand']
  NaNs:        [0 0]
  Min demand:  0.002795  (must be >= 0)
  Max demand:  1.000000  (must be <= 1)

   Index  demand
0      0  0.0411
1      1  0.0312
2      2  0.0112
3      3  0.0363
4      4  0.0581
5      5  0.0112
6      6  0.0335
7      7  0.1126
8      8  0.0286
9      9  0.0682
